### General QUBO:
$$x^T Qx+q^T x+c$$
### Symmetrize Q:
$$Q\to\frac{Q+Q^T}{2}$$
### Transform to spin variables: 
$$
\begin{aligned}
x & \to\frac{u-s}{2}, \quad u=(1,1,\dots,1) \\
&=\frac{1}{4} (u-s)^T Q(u-s)+\frac{1}{2} q^T (u-s)+c \\
&=\frac{1}{4} u^T Qu-\frac{1}{4} u^T Qs-\frac{1}{4} s^T Qu+\frac{1}{4} s^T Qs+\frac{1}{2} q^T u-\frac{1}{2} q^T s+c
\end{aligned}
$$
### Separate terms:
*	Quadratic:
$$\frac{1}{4} s^T Qs$$
*	Linear:
$$-\frac{1}{4} u^T Qs-\frac{1}{4} s^T Qu-\frac{1}{2} q^T s=-\frac{1}{4} u^T (Q^T+Q)s-\frac{1}{2} q^T s=-\frac{1}{2} (Qu+q)^T s$$
*	Constant:
$$\frac{1}{4} u^T Qu+\frac{1}{2} q^T u+c$$


In [ ]:
import numpy as np


def qubo_to_ising(Q: np.ndarray, q: np.ndarray | None = None, c: float = 0) -> tuple[np.ndarray, np.ndarray, float]:
    Q = np.array(Q, float)
    if q is None:
        q = np.zeros(Q.shape[0])
    else:
        q = np.array(q, float)
    n = q.shape[0]
    assert Q.shape == (n, n), "Q must be a square matrix compatible with q"
    # symmetrize Q
    Q = (Q + Q.T) / 2
    # move diagonal to the linear terms, require only Z gate
    diagQ = np.diag(Q)
    J = Q / 4
    np.fill_diagonal(J, 0)
    h = -(np.sum(Q, axis=1) + diagQ + q) / 2
    bias = np.sum(Q) / 4 + np.sum(q) / 2 + c
    return (J, h, bias)

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter


def ising_hamiltonian(J: np.ndarray, h: np.ndarray, gamma_param: Parameter) -> QuantumCircuit:
    n = h.shape[0]
    qc = QuantumCircuit(n)
    for qubit in range(n):
        qc.rz(2 * h[qubit] * gamma_param, qubit)
    for qubit1 in range(n):
        for qubit2 in range(qubit1):
            qc.rzz(2 * J[qubit1, qubit2] * gamma_param, qubit1, qubit2)
    return qc


def mixing_hamiltonian(n: int, beta_param: Parameter) -> QuantumCircuit:
    qc = QuantumCircuit(n)
    for i in range(n):
        qc.rx(2 * beta_param, i)
    return qc

In [ ]:
from qiskit.circuit import ParameterVector
def qaoa_circuit(J: np.ndarray, h: np.ndarray, layers: int) -> QuantumCircuit:
    n = h.shape[0]
    gamma = ParameterVector("gamma", layers)
    beta = ParameterVector("beta", layers)
    qc = QuantumCircuit(n)
    # initialize states to superposition
    qc.h(range(n))
    for layer in range(layers):
        qc.compose(ising_hamiltonian(J, h, gamma[layer]), inplace=True)
        qc.compose(mixing_hamiltonian(n, beta[layer]), inplace=True)
    return qc

In [ ]:
from qiskit import transpile
from qiskit.primitives import BaseEstimatorV2, BaseSamplerV2
from scipy.optimize import minimize
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager


def build_sparse_pauli_op(J: np.ndarray, h: np.ndarray):
    n = h.shape[0]
    pauli_list = []
    for i in range(n):
        if h[i] != 0:
            pauli = ["I"] * n
            pauli[i] = "Z"
            pauli_list.append(("".join(pauli[::-1]), h[i]))
    for i in range(n):
        for j in range(i):
            if J[i, j] != 0:
                pauli = ["I"] * n
                pauli[i] = "Z"
                pauli[j] = "Z"
                pauli_list.append(("".join(pauli[::-1]), J[i, j]))
    return SparsePauliOp.from_list(pauli_list)


def qaoa(J: np.ndarray, h: np.ndarray, gamma0: list[float], beta0: list[float], estimator: BaseEstimatorV2, sampler: BaseSamplerV2, optimizer: str = "COBYLA", bias: float = 0):
    assert len(gamma0) == len(beta0), "parameters must have same length"
    layers = len(gamma0)
    n = h.shape[0]
    qc = qaoa_circuit(J, h, layers)

    backend = getattr(estimator, "backend", None)
    if backend is None:
        backend = getattr(estimator, "_backend", None)
    print(f"Using backend: {backend}")

    pm = generate_preset_pass_manager(optimization_level=3, backend=backend)
    isa_qc = pm.run(qc)

    observable = build_sparse_pauli_op(J, h)
    isa_observable = observable.apply_layout(isa_qc.layout)

    history = []

    def objective(params: list[float]) -> float:
        pub = (isa_qc, isa_observable, params)
        res = estimator.run([pub]).result()[0]
        cost = float(res.data.evs) + bias
        history.append({"gamma": params[:layers], "beta": params[layers:], "cost": cost})
        return cost

    x0 = np.concatenate([gamma0, beta0])
    result = minimize(objective, x0=x0, method=optimizer)

    qc_meas = qc.measure_all(False)
    isa_meas = pm.run(qc_meas)

    pub_sample = (isa_meas, result.x)
    final_counts = sampler.run([pub_sample], shots=4096).result()[0].data.meas.get_counts()
    sorted_counts = dict(sorted(final_counts.items(), key=lambda item: item[1], reverse=True))

    best_bitstring = list(sorted_counts.keys())[0]
    best_x = np.array([int(b) for b in reversed(best_bitstring)])

    return {
        "x": best_x,
        "bitstring": best_bitstring,
        "cost": result.fun,
        "gammas": result.x[:layers],
        "betas": result.x[layers:],
        "counts": sorted_counts,
        "history": history,
        "converged": result.success,
    }

### QUBO formulation of Vertex Cover
$$
\begin{aligned}
  H &= \sum_{\left(u,v\right) \in E}{\left(1-x_v\right)\left(1-x_u\right)}+P\sum_{v\in V}{x_v},\quad 0<P<1 \\
    &= \sum_{\left(u,v\right) \in V^2}{E_{uv} \left(1-x_v\right)\left(1-x_u\right)} + P\sum_{v\in V}{x_v}, \quad E_{uv} = 1 \iff \left(u,v\right) \in E, \quad E \space \text{symmetric} \\
    &= \left(\vec{1} - \vec{x}\right)^T E \left(\vec{1} - \vec{x}\right) + P\vec{1}^T\vec{x} \\
    &= \vec{1}^TE\vec{1} - \vec{1}^T E \vec{x} - \vec{x}^T E \vec{1} + \vec{x}^T E \vec{x} + P\vec{1}^T\vec{x} \\
    &= \vec{1}^TE\vec{1} - \vec{1}^T\left( E +  E^T + PI\right)\vec{x} + \vec{x}^TE\vec{x} \\
    &= \vec{1}^TE\vec{1} - \vec{1}^T\left(2E + PI\right)\vec{x} + \vec{x}^TE\vec{x} \\
    &= \vec{1}^TE\vec{1} - \left(\left(2E + PI\right)\vec{1}\right)^T\vec{x} + \vec{x}^TE\vec{x}
\end{aligned}
$$

Quadratic term: $$\vec{x}^TE\vec{x} \Rightarrow Q = E$$

Linear term: $$ - \left(\left(2E + PI\right)\vec{1}\right)^T\vec{x} \Rightarrow q = -\left(2E + PI\right)\vec{1}$$

Constant term: $$ c = \vec{1}^TE\vec{1} $$


In [ ]:
from rustworkx import PyGraph, adjacency_matrix


class VertexCover:
    def __init__(self, graph: PyGraph, P: float = .5) -> None:
        assert 0 < P < 1, "Penalty must be in range (0,1)"
        self._graph = graph
        self._n = graph.num_nodes()
        edges = graph.edge_list()
        self._E = adjacency_matrix(graph).astype(np.int8)
        self._P = P
        self._qubo = None
        self._ising = None

    @property
    def qubo(self) -> tuple[np.ndarray, np.ndarray, float]:
        if self._qubo is None:
            Q = self._E.copy()
            q = self._P - 2 * self._E.sum(axis=1).flatten()
            c = float(self._E.sum())
            self._qubo = (Q, q, c)
        return self._qubo

    @property
    def ising(self) -> tuple[np.ndarray, np.ndarray, float]:
        if self._ising is None:
            self._ising = qubo_to_ising(*self.qubo)
        return self._ising

    def solve(self, layers: int = 3, optimizer: str = "COBYLA", estimator: BaseEstimatorV2 | None = None, sampler: BaseSamplerV2 | None = None, backend=None, seed: int | None = None) -> dict:
        if estimator is None or sampler is None:
            if backend is not None:
                backend_classname = type(backend).__name__
                if "IBM" in backend_classname or "BackendV2" in backend_classname:
                    from qiskit_ibm_runtime import EstimatorV2, SamplerV2
                    estimator = EstimatorV2(backend=backend) if estimator is None else estimator
                    sampler = SamplerV2(backend=backend) if sampler is None else sampler
                else:
                    from qiskit_aer.primitives import EstimatorV2, SamplerV2
                    estimator = EstimatorV2(backend=backend)
                    sampler = SamplerV2(backend=backend)
            else:
                from qiskit_aer.primitives import EstimatorV2, SamplerV2
                estimator = EstimatorV2()
                sampler = SamplerV2()

        J, h, bias = self.ising

        print("--------------------------------------------------")
        print(f"Graph Size:          {self._n} nodes, {len(self._graph.edge_list())} edges")
        print(f"Penalty parameter P: {self._P}")
        print(f"QAOA Layers (p):     {layers}")
        print(f"Classical Optimizer: {optimizer}")
        print("--------------------------------------------------")

        if seed is not None:
            np.random.seed(seed)
        gamma0 = list(np.random.uniform(-np.pi, np.pi, layers))
        beta0 = list(np.random.uniform(-np.pi, np.pi, layers))

        qaoa_result = qaoa(
            J=J,
            h=h,
            gamma0=gamma0,
            beta0=beta0,
            estimator=estimator,
            sampler=sampler,
            optimizer=optimizer,
            bias=bias,
        )

        all_counts = qaoa_result["counts"]
        valid_covers = []

        for bitstring, count in all_counts.items():
            solution_vector = np.array([int(b) for b in reversed(bitstring)])

            is_valid = True
            for u, v in self._graph.edge_list():
                if solution_vector[u] == 0 and solution_vector[v] == 0:
                    is_valid = False
                    break

            if is_valid:
                cover_size = int(sum(solution_vector))
                valid_covers.append({
                    "bitstring": bitstring,
                    "vector": solution_vector,
                    "shots": count,
                    "size": cover_size,
                })

        valid_covers.sort(key=lambda item: item["shots"], reverse=True)
        valid_covers.sort(key=lambda item: item["size"])

        print("\n--- PERFORMANCE & RESULTS REPORT ---")
        print(f"Optimization Convergence Status: {qaoa_result['converged']}")
        print(f"Final Estimated Min Energy:     {qaoa_result['cost']:.4f}")

        if not valid_covers:
            print("\nNo valid vertex covers were sampled in this execution.")
        else:
            print(f"\nFound {len(valid_covers)} distinct valid vertex covers among the samples.")
            print(f"Minimum Cover Size discovered: {valid_covers[0]['size']}\n")

            print(f"{'Rank':<6} | {'Bitstring':<12} | {'Cover Size':<10} | {'Shots Count':<12}")
            print("-" * 50)
            for rank, cover in enumerate(valid_covers[:10]):
                print(f"{rank + 1:<6} | {cover['bitstring']:<12} | {cover['size']:<10} | {cover['shots']:<12}")

        qaoa_result["valid_covers"] = valid_covers
        return qaoa_result

### QUBO formulation of Number Partitioning

The Lucas paper gives the Hamiltonian as $H = (\sum_{i=1}^n {n_i s_i})^2$. To transform it to the QUBO formulation, use the transformation $s_i = 2x_i -1$.
$$
\begin{aligned}
  H &= (\sum_{i}{n_is_i})^2 \\
    &= (\sum_{i} n_i-2\sum_{i}{n_ix_i})^2 \\
    &= (\sum_{i} n_i)^2-4(\sum_{i} n_i)(\sum_{i} x_i)+4\sum_{i}\sum_{j}{n_in_jx_ix_j} \\ 
    &= A^2 -4A{\vec{1}}^T\vec{x} + 4{\vec{x}}^TN\vec{x}, \quad A = \sum_{i} n_i, \quad N = \vec{n}\vec{n}^T
\end{aligned}
$$

Quadratic term: $$ 4\vec{x}^T N \vec{x} \Rightarrow Q = 4N $$

Linear term: $$ -4A{\vec{1}}^T\vec{x} \Rightarrow q = -4A\vec{1} $$

Constant term: $$ c = A^2 $$

In [ ]:
from numbers import Real


class NumberPartition:
    def __init__(self, n: set[Real]) -> None:
        self._n = np.sort(list(n))
        self._qubo = None
        self._ising = None

    @property
    def qubo(self) -> tuple[np.ndarray, np.ndarray, float]:
        if self._qubo == None:
            N = np.outer(self._n, self._n)
            Q = 4 * N
            A = np.sum(self._n)
            q = -4 * A + np.zeros_like(self._n)
            c = A ** 2
            self._qubo = (Q, q, c)
        return self._qubo

    @property
    def ising(self) -> tuple[np.ndarray, np.ndarray, float]:
        if self._ising is None:
            self._ising = qubo_to_ising(*self.qubo)
        return self._ising

    def solve(self, layers: int = 3, optimizer: str = "COBYLA", estimator: BaseEstimatorV2 | None = None, sampler: BaseSamplerV2 | None = None, backend=None, seed: int | None = None) -> dict:
        if estimator is None or sampler is None:
            if backend is not None:
                backend_classname = type(backend).__name__
                if "IBM" in backend_classname or "BackendV2" in backend_classname:
                    from qiskit_ibm_runtime import EstimatorV2, SamplerV2
                    estimator = EstimatorV2(backend=backend) if estimator is None else estimator
                    sampler = SamplerV2(backend=backend) if sampler is None else sampler
                else:
                    from qiskit_aer.primitives import EstimatorV2, SamplerV2
                    estimator = EstimatorV2(backend=backend)
                    sampler = SamplerV2(backend=backend)
            else:
                from qiskit_aer.primitives import EstimatorV2, SamplerV2
                estimator = EstimatorV2()
                sampler = SamplerV2()

        J, h, bias = self.ising

        print("--------------------------------------------------")
        print(f"Problem Size:        {self._n.shape[0]} numbers")
        print(f"QAOA Layers (p):     {layers}")
        print(f"Classical Optimizer: {optimizer}")
        print("--------------------------------------------------")

        if seed is not None:
            np.random.seed(seed)
        gamma0 = list(np.random.uniform(-np.pi, np.pi, layers))
        beta0 = list(np.random.uniform(-np.pi, np.pi, layers))

        qaoa_result = qaoa(
            J=J,
            h=h,
            gamma0=gamma0,
            beta0=beta0,
            estimator=estimator,
            sampler=sampler,
            optimizer=optimizer,
            bias=bias,
        )

        all_counts = qaoa_result["counts"]

        # Dictionary to consolidate symmetric partitions
        consolidated_solutions = {}

        for bitstring, count in all_counts.items():
            if not bitstring:
                continue

            # Normalize symmetric bitstrings: if it starts with '1', flip it to its complement
            # For example, '1001' becomes '0110'
            if bitstring[0] == '1':
                normalized_bitstring = "".join('1' if b == '0' else '0' for b in bitstring)
            else:
                normalized_bitstring = bitstring

            if normalized_bitstring in consolidated_solutions:
                consolidated_solutions[normalized_bitstring]["shots"] += count
            else:
                # Qiskit bitstrings are ordered from right-to-left (MSB to LSB).
                # Reversing maps bitstring index directly to self._n indices.
                solution_vector = np.array([int(b) for b in reversed(normalized_bitstring)])

                # Map 0 -> Subset A, 1 -> Subset B
                subset_A = self._n[solution_vector == 0]
                subset_B = self._n[solution_vector == 1]

                # Calculate the absolute difference between the sums of the two subsets
                diff = abs(np.sum(subset_A) - np.sum(subset_B))

                consolidated_solutions[normalized_bitstring] = {
                    "bitstring": normalized_bitstring,
                    "subset_A": subset_A.tolist(),
                    "subset_B": subset_B.tolist(),
                    "difference": diff,
                    "shots": count
                }

        # Convert dictionary back to a list
        parsed_solutions = list(consolidated_solutions.values())

        # Sort primarily by the smallest subset difference, and secondarily by highest frequency (shots)
        parsed_solutions.sort(key=lambda item: (item["difference"], -item["shots"]))

        print("\n--- PERFORMANCE & RESULTS REPORT ---")
        print(f"Optimization Convergence Status: {qaoa_result['converged']}")
        print(f"Final Estimated Min Energy:      {qaoa_result['cost']:.4f}")
        print(f"\nBest Partition Difference Discovered: {parsed_solutions[0]['difference']}\n")

        # Format layout to present the detailed subset divisions clearly
        header = f"{'Rank':<5} | {'Bitstring':<10} | {'Diff':<6} | {'Shots':<6} | {'Subset A / Subset B'}"
        print(header)
        print("-" * len(header))

        for rank, sol in enumerate(parsed_solutions[:10]):
            subsets_str = f"{sol['subset_A']} vs {sol['subset_B']}"
            print(f"{rank + 1:<5} | {sol['bitstring']:<10} | {sol['difference']:<6.2f} | {sol['shots']:<6} | {subsets_str}")

        qaoa_result["partitions"] = parsed_solutions
        return qaoa_result


num = NumberPartition({23, -666, 1 / 7, 2007, 12349, 0.8, 456, 999, 626, 8})
num.solve()